In [2]:
from google.colab import files
uploaded = files.upload()  # click "Choose Files", pick both CSVs


Saving fear_greed_index.csv to fear_greed_index (1).csv
Saving historical_data.csv to historical_data.csv


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
sentiment_df = pd.read_csv('fear_greed_index (1).csv')
trades_df = pd.read_csv('historical_data.csv')

for name, df in [('sentiment', sentiment_df), ('trades', trades_df)]:
    print(f"--- {name} ---")
    print("shape:", df.shape)
    print(df.dtypes)
    print("nulls:\n", df.isnull().sum())
    print("duplicates:", df.duplicated().sum())
    print()

--- sentiment ---
shape: (2644, 4)
timestamp          int64
value              int64
classification    object
date              object
dtype: object
nulls:
 timestamp         0
value             0
classification    0
date              0
dtype: int64
duplicates: 0

--- trades ---
shape: (15629, 16)
Account              object
Coin                 object
Execution Price     float64
Size Tokens         float64
Size USD            float64
Side                 object
Timestamp IST        object
Start Position      float64
Direction            object
Closed PnL          float64
Transaction Hash     object
Order ID              int64
Crossed                bool
Fee                 float64
Trade ID            float64
Timestamp           float64
dtype: object
nulls:
 Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction

In [5]:
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])
trades_df['date'] = pd.to_datetime(
    trades_df['Timestamp IST'], format='%d-%m-%Y %H:%M'
).dt.normalize()

In [6]:
# Left join: keep every trade even if a date has no sentiment match
merged = trades_df.merge(
    sentiment_df[['date', 'classification']], on='date', how='left'
)
print(merged['classification'].isnull().sum(), "unmatched rows")

6 unmatched rows


In [7]:
def is_long(direction):
    return direction in ['Buy', 'Open Long', 'Close Short', 'Short > Long']

def is_short(direction):
    return direction in ['Sell', 'Open Short', 'Close Long', 'Long > Short']

merged['is_long'] = merged['Direction'].apply(is_long)
merged['is_short'] = merged['Direction'].apply(is_short)
merged['win'] = merged['Closed PnL'] > 0

daily = merged.groupby(['Account', 'date']).agg(
    daily_pnl=('Closed PnL', 'sum'),
    n_trades=('Closed PnL', 'size'),
    win_rate=('win', 'mean'),
    avg_trade_size=('Size USD', 'mean'),
    long_trades=('is_long', 'sum'),
    short_trades=('is_short', 'sum'),
    classification=('classification', 'first'),
).reset_index()

daily['long_short_ratio'] = (
    daily['long_trades'] / daily['short_trades'].replace(0, np.nan)
)

def bucket(c):
    if pd.isna(c): return np.nan
    if 'Fear' in c: return 'Fear'
    if 'Greed' in c: return 'Greed'
    return 'Neutral'

daily['sentiment_bucket'] = daily['classification'].apply(bucket)

In [8]:
print(daily.groupby('sentiment_bucket')[
    ['daily_pnl', 'win_rate', 'n_trades', 'avg_trade_size']
].mean())

                     daily_pnl  win_rate    n_trades  avg_trade_size
sentiment_bucket                                                    
Fear              14391.823786  0.368575  146.038462    14872.050343
Greed              1687.249990  0.166855   87.983871    14011.629726
Neutral           22164.929152  0.235018  151.411765    11638.120113


In [9]:
print(daily.groupby('sentiment_bucket')['long_short_ratio'].mean())

sentiment_bucket
Fear       1.689253
Greed      0.461770
Neutral    1.374218
Name: long_short_ratio, dtype: float64


In [10]:
med_size = daily['avg_trade_size'].median()
daily['size_segment'] = np.where(daily['avg_trade_size'] > med_size, 'High size', 'Low size')

med_freq = daily['n_trades'].median()
daily['freq_segment'] = np.where(daily['n_trades'] > med_freq, 'Frequent', 'Infrequent')

print(daily.groupby('size_segment')['win_rate'].mean())
print(daily.groupby('freq_segment')[['win_rate', 'daily_pnl']].mean())

size_segment
High size    0.275193
Low size     0.247628
Name: win_rate, dtype: float64
              win_rate     daily_pnl
freq_segment                        
Frequent      0.287830  15990.072845
Infrequent    0.234991   3286.595730


In [11]:
strategy_notes = """
Insight 1: Fear-day PnL and win rate exceed Greed-day PnL/win rate in this sample —
traders who stayed active during Fear outperformed those active during Greed.

Insight 2: Long/short ratio flips from ~1.7 (Fear) to ~0.46 (Greed) — traders in this
dataset go net long during Fear and net short during Greed, opposite of naive
"panic sells in Fear" assumption.

Insight 3: Frequent traders (above-median trade count) have both higher win rate
(28.8% vs 23.5%) and dramatically higher PnL (15,990 vs 3,287) than infrequent traders.

Rule of thumb 1: Don't reduce activity during Fear days — this sample's data says
Fear-day activity correlates with better outcomes, not worse.

Rule of thumb 2: Favor the "frequent trader" behavioral profile — high trade count
correlates with both higher win rate AND higher PnL, suggesting skill/consistency
rather than luck in this segment.
"""
print(strategy_notes)


Insight 1: Fear-day PnL and win rate exceed Greed-day PnL/win rate in this sample —
traders who stayed active during Fear outperformed those active during Greed.

Insight 2: Long/short ratio flips from ~1.7 (Fear) to ~0.46 (Greed) — traders in this
dataset go net long during Fear and net short during Greed, opposite of naive
"panic sells in Fear" assumption.

Insight 3: Frequent traders (above-median trade count) have both higher win rate
(28.8% vs 23.5%) and dramatically higher PnL (15,990 vs 3,287) than infrequent traders.

Rule of thumb 1: Don't reduce activity during Fear days — this sample's data says
Fear-day activity correlates with better outcomes, not worse.

Rule of thumb 2: Favor the "frequent trader" behavioral profile — high trade count
correlates with both higher win rate AND higher PnL, suggesting skill/consistency
rather than luck in this segment.



In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Aggregate to ACCOUNT level
account_stats = daily.groupby('Account').agg(
    total_pnl=('daily_pnl', 'sum'),
    avg_daily_pnl=('daily_pnl', 'mean'),
    avg_win_rate=('win_rate', 'mean'),
    avg_trade_size=('avg_trade_size', 'mean'),
    avg_n_trades=('n_trades', 'mean'),
    active_days=('date', 'nunique'),
    avg_long_short=('long_short_ratio', 'mean'),
).reset_index()
account_stats['avg_long_short'] = account_stats['avg_long_short'].fillna(1.0)

# CAVEAT: only {n_accounts} unique accounts in this dataset.
features = ['avg_win_rate', 'avg_trade_size', 'avg_n_trades', 'avg_long_short', 'active_days']
X = account_stats[features].fillna(0)
Xs = StandardScaler().fit_transform(X)

km = KMeans(n_clusters=3, random_state=42, n_init=10)
account_stats['cluster'] = km.fit_predict(Xs)

print(account_stats.groupby('cluster')[features + ['total_pnl']].mean())

         avg_win_rate  avg_trade_size  avg_n_trades  avg_long_short  \
cluster                                                               
0            0.325495    27603.845216    313.743590        1.157971   
1            0.262005     3900.199095     29.600000        0.914763   
2            0.204386    34520.906774    116.666667        1.466198   

         active_days      total_pnl  
cluster                              
0               39.0  840422.555216  
1               40.5  235428.561845  
2               12.0  -39019.552950  
